# 03 – Preprocessing, token lengths and vocabulary

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ilaria-cesa/Txt_lab/blob/main/notebooks/03_preprocessing_and_lengths.ipynb)

Work in progress. First step of the **classification** task, on the training set with the cap per class (notebook 02):

1. **light cleaning** (common to every model) of *title + abstract*;
2. **token lengths** for the tokenizers of the transformers we may use: how many papers exceed 256 and 512 tokens?
3. **domain stop words**: which frequent words say nothing about the class?
4. **vocabulary table**: how much does each step shrink the vocabulary (Exercise 1.2 of Lab 1.1)?
5. **normalization for the sparse models**: nothing / stemming / lemmatization, compared with TF-IDF + linear SVM on the validation set.

The rules live in `src/preprocessing.py`. Parameters still to be decided are marked `TODO`.

## 1. Setup and parameters

In [ ]:
import os, sys, subprocess, pathlib, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/Txt_lab"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "https://github.com/ilaria-cesa/Txt_lab", REPO], check=True)
    else:
        subprocess.run(["git", "-C", REPO, "pull"], check=True)      # picks up files pushed after the first clone
    SRC = f"{REPO}/src"
    DATA_DIR = "/content/drive/MyDrive/Txt_lab_data"
else:
    SRC = str(pathlib.Path("..") / "src")
    DATA_DIR = "../data"
sys.path.insert(0, SRC)

SPLIT_DIR = f"{DATA_DIR}/cleaning_splitting"     # written by notebook 02
OUT_DIR = f"{DATA_DIR}/preprocessing"
os.makedirs(OUT_DIR, exist_ok=True)

import label_mapping as lm, preprocessing as pp   # if you pushed new versions: Runtime > Restart session

In [ ]:
# --- Parameters ----------------------------------------------------------------------------
TRAIN_FILE = "train_cap40000.csv.gz"   # training set with the cap per class (notebook 02, section 7)
VAL_FILE = "val.csv.gz"
VAL_SAMPLE = 30000        # validation papers used in section 6 (None = all); random, natural distribution
LENGTH_SAMPLE = 30000     # training papers used for the token length analysis (None = all)
VOCAB_SAMPLE = None       # training papers used for the vocabulary table (None = all; the table takes a few minutes)
N_PROCESS = 2             # processes for the spaCy lemmatizer
SEED = 42

# tokenizers whose lengths we measure (the model id decides the tokenizer)
TOKENIZERS = {
    "SciBERT (scivocab)": "allenai/scibert_scivocab_uncased",
    "SPECTER2 base": "allenai/specter2_base",
    "ModernBERT-base": "answerdotai/ModernBERT-base",
    "bge-base-en-v1.5": "BAAI/bge-base-en-v1.5",
    "e5-base-v2": "intfloat/e5-base-v2",
}

In [ ]:
train = pd.read_csv(f"{SPLIT_DIR}/{TRAIN_FILE}", dtype={"arxiv_id": str})
val = pd.read_csv(f"{SPLIT_DIR}/{VAL_FILE}", dtype={"arxiv_id": str})
if VAL_SAMPLE:
    val = val.sample(min(VAL_SAMPLE, len(val)), random_state=SEED)
train["text"] = pp.build_texts(train)      # light cleaning: the same input for every model
val["text"] = pp.build_texts(val)
print("train", train.shape, "| validation sample", val.shape)
print(train["technique_label"].value_counts().to_string())

In [ ]:
# what the light cleaning does: a paper with a formula, before and after
has_math = train["abstract"].str.contains(r"\$", regex=True)
ex = train[has_math].iloc[0] if has_math.any() else train.iloc[0]
print("BEFORE:", ex["title"], "|", ex["abstract"][:400], "\n")
print("AFTER: ", ex["text"][:400])

## 2. Token lengths

The transformers read at most 512 tokens (ModernBERT many more). A *token* is a sub-word piece, so there are more tokens than words, and each
tokenizer splits scientific vocabulary differently. For every tokenizer we count how many papers exceed **256** (the length we may use to fine-tune
on a free Colab GPU) and **512** (the limit of the BERT-style models). Stop words are not removed: a transformer reads natural text.

In [ ]:
from transformers import AutoTokenizer

len_sample = train.sample(min(LENGTH_SAMPLE, len(train)), random_state=SEED) if LENGTH_SAMPLE else train
texts_len = len_sample["text"].tolist()
all_len, stats = {}, {}
for name, model_id in TOKENIZERS.items():
    try:
        tok = AutoTokenizer.from_pretrained(model_id)
    except Exception as e:
        print(f"{name}: tokenizer not available ({str(e)[:80]})"); continue
    ids = tok(texts_len, add_special_tokens=True, truncation=False)["input_ids"]
    all_len[name] = np.array([len(x) for x in ids])
    stats[name] = pp.token_length_stats(all_len[name], limits=(256, 512))
words = len_sample["text"].str.split().str.len()
stats_df = pd.DataFrame(stats).T
stats_df["tokens per word"] = [round(float(all_len[n].mean() / words.mean()), 2) for n in stats_df.index]
print("words per paper: mean %.0f, max %d" % (words.mean(), words.max()))
stats_df.to_csv(f"{OUT_DIR}/token_length_stats.csv")
stats_df

In [ ]:
first = next(iter(all_len))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, lens in all_len.items():
    ax[0].hist(lens, bins=60, histtype="step", label=name, range=(0, 700))
for lim in (256, 512):
    ax[0].axvline(lim, color="k", ls="--", lw=1)
ax[0].set_xlabel("tokens (title + abstract)"); ax[0].set_ylabel("papers"); ax[0].legend(fontsize=7)
ax[0].set_title("Token length distribution")
over = pd.Series(all_len[first] > 256).groupby(len_sample["technique_label"].values).mean().mul(100)
over.sort_values().plot.barh(ax=ax[1], title=f"% of papers above 256 tokens per class ({first})")
plt.tight_layout(); plt.show()

Reading guide: if few papers exceed 512, truncation is enough and chunking would change almost nothing (we can state it in the report as an ablation with a
negligible effect). If many exceed 256, fine-tuning at 256 tokens cuts a large part of the text: look at *which* classes lose most (right plot).

Model-specific input formats (e.g. SPECTER2 expects title and abstract joined by the separator token, e5 expects a `query: ` prefix) are added in the embedding notebook, because they
are not part of the common cleaning.

## 3. Frequent words and domain stop words

Standard English stop words (NLTK list, as in the labs) **without the negations** (`not`, `no`, `nor`, `...n't`). Then, on the **training set**, the words that appear in most papers.
The table also shows, for each of them, the % of papers of each class that contain it: a word with a small `spread` between classes says nothing about the class and can be a domain stop word;
a word with a large spread (for example `image`, `language`) is signal and must stay.

In [ ]:
STOPWORDS = pp.build_stopwords()          # NLTK English, negations kept
print(len(STOPWORDS), "standard stop words | negations kept:",
      all(w not in STOPWORDS for w in ("not", "no", "nor")))
train_tokens = [pp.preprocess_sparse(t, STOPWORDS) for t in train["text"]]
dfreq = pp.document_frequency(train_tokens, top=100)

sets = [set(t) for t in train_tokens]
labels = train["technique_label"].values
presence = pd.DataFrame({tok: [tok in s for s in sets] for tok in dfreq["token"]})
by_class = presence.groupby(labels).mean().T.mul(100)
table = dfreq.set_index("token").join(by_class.round(0).astype(int))
table["spread"] = (by_class.max(axis=1) - by_class.min(axis=1)).round(0).astype(int)
table

In [ ]:
# TODO: choose the domain stop words after reading the table above (low `spread`, no topical meaning).
# The seed list is only a starting point: it is compared with and without in section 6.
DOMAIN_STOPWORDS = list(pp.DOMAIN_STOPWORDS_SEED)
print(DOMAIN_STOPWORDS)
ALLSTOP = STOPWORDS | set(DOMAIN_STOPWORDS)

## 4. Lemmatization (spaCy)

Lemmatization needs the context (the part of speech), so it runs on the light-cleaned text and **before** the stop words. The result is cached in Google Drive and does not depend on the stop word lists:
changing `DOMAIN_STOPWORDS` does not require running it again.

In [ ]:
import pickle
nlp = pp.load_spacy()
cache = pathlib.Path(OUT_DIR) / f"lemmas_{TRAIN_FILE}_{len(train)}_{len(val)}_{SEED}.pkl"
if cache.exists():
    train_lem, val_lem = pickle.load(open(cache, "rb"))
    print("lemmas loaded from", cache)
else:
    t0 = time.time()
    train_lem = pp.lemma_tokens(train["text"], nlp, n_process=N_PROCESS)       # only numbers / 1-char tokens filtered
    val_lem = pp.lemma_tokens(val["text"], nlp, n_process=N_PROCESS)
    pickle.dump((train_lem, val_lem), open(cache, "wb"))
    print(f"lemmatization: {(time.time() - t0) / 60:.1f} minutes")

## 5. Vocabulary table

Vocabulary = number of different tokens. In TF-IDF every one is a column, so a smaller vocabulary means a smaller and faster matrix and fewer very rare words. Steps are cumulative; the *side* rows branch from the last common step.

In [ ]:
vs = train.sample(VOCAB_SAMPLE, random_state=SEED) if VOCAB_SAMPLE else train
vs_lem = [pp.filter_tokens(l, ALLSTOP, drop_numbers=False, min_len=1) for l in
          (train_lem if not VOCAB_SAMPLE else [train_lem[i] for i in train.index.get_indexer(vs.index)])]
vocab_table = pp.vocabulary_steps(vs["text"], STOPWORDS, DOMAIN_STOPWORDS, lemmas=vs_lem)
vocab_table.to_csv(f"{OUT_DIR}/vocabulary_steps.csv", index=False)
vocab_table

## 6. Normalization for the sparse models: does it matter?

TF-IDF (unigrams + bigrams, `min_df=5`, sublinear tf) + linear SVM with balanced class weights, fitted on the training set and scored on the validation sample. Only the preprocessing changes between rows.
The vectorizer is fitted on the training set only. Macro-F1 is the main metric; accuracy is shown only to compare.

Differences of a few tenths of a point are probably noise (the smallest classes have a few hundred validation papers): if two variants are equivalent, prefer the simpler one.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score

VARIANTS = {
    "A  tokens only (no stop words)": dict(stop=frozenset(), stem=False, lemma=False),
    "B  + standard stop words":       dict(stop=STOPWORDS,   stem=False, lemma=False),
    "C  + domain stop words":         dict(stop=ALLSTOP,     stem=False, lemma=False),
    "D  C + Porter stemming":         dict(stop=ALLSTOP,     stem=True,  lemma=False),
    "E  C + lemmatization (spaCy)":   dict(stop=ALLSTOP,     stem=False, lemma=True),
}

def tokens_for(v, texts, lemmas):
    if v["lemma"]:
        return [pp.filter_tokens(l, v["stop"], drop_numbers=False, min_len=1) for l in lemmas]
    return [pp.preprocess_sparse(t, v["stop"], stem=v["stem"]) for t in texts]

classes = lm.TECHNIQUE_CLASSES
y_tr, y_va = train["technique_label"].values, val["technique_label"].values
rows, per_class = [], {}
for name, v in VARIANTS.items():
    t0 = time.time()
    tr = [" ".join(t) for t in tokens_for(v, train["text"], train_lem)]
    va = [" ".join(t) for t in tokens_for(v, val["text"], val_lem)]
    vec = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False,
                          ngram_range=(1, 2), min_df=5, sublinear_tf=True)
    X_tr = vec.fit_transform(tr)
    X_va = vec.transform(va)
    clf = LinearSVC(class_weight="balanced").fit(X_tr, y_tr)
    pred = clf.predict(X_va)
    rows.append({"variant": name, "features": X_tr.shape[1],
                 "macro-F1": round(100 * f1_score(y_va, pred, average="macro", labels=classes, zero_division=0), 2),
                 "accuracy": round(100 * accuracy_score(y_va, pred), 2),
                 "minutes": round((time.time() - t0) / 60, 1)})
    per_class[name] = dict(zip(classes, (100 * f1_score(y_va, pred, average=None, labels=classes, zero_division=0)).round(1)))
results = pd.DataFrame(rows).set_index("variant")
results.to_csv(f"{OUT_DIR}/normalization_comparison.csv")
display(results)
pd.DataFrame(per_class).T.rename_axis("F1 per class")

## 7. Decisions (to fill in after reading the tables)

| Choice | Decision | Evidence |
|---|---|---|
| Domain stop words | TODO | section 3 + variants B/C |
| Stemming / lemmatization / nothing | TODO | section 5 (vocabulary) + variants C/D/E |
| Truncation or chunking for the transformers | TODO | section 2 (% above 256 and 512 tokens) |

## Notes

- Everything that is learned from the data (frequent words, vocabulary, IDF) comes from the **training** set; the validation sample is only transformed and scored.
- `categories` and `n_categories` contain the label: **never use them as features**.
- The text for the transformers is the light-cleaned one: no lowercasing, no stop word removal, no stemming or lemmatization.